# AR identity / mirror diagnostics (notebook 17): what are Transformer AR and Looped AR doing wrong?

**Diagnostic only.**
* Reads the SAVED free-running predictions of the identity-format AR runs: fixed S=5 supports 02/023/0234/02345 and
  the notebook-16 slot-count runs S=2/3/4/6, plus optional variable_length history.
* No training, fine-tuning, re-evaluation or new inputs. CPU is enough.

**Method.**
* Every singleton, the selected same-target controls and all 729,000 six-step records per S=6 model are matched EXACTLY
  to the 288 legal spatial transforms. Ambiguous match sets are kept, never broken arbitrarily.
* Wrong legal outputs are split into extra output-C / extra input-C / other residual classes.
* Also: same-target pairs (A~ACC, B~BCC, C~CCC, C~CCCCC), held-out source-position copying maps, six-step strata by
  number of C tokens, and placement / slot-count / format comparisons.

**Questions answered in the final evidence table.**
1. Is singleton C predicted as identity, another legal transform, or a non-legal grid?
2. Under train023, do A/B failures share an extra-mirror error with C?
3. Can ACC/BCC/CCC be correct while A/B/C fail on the same grid?
4. Do S=6 six-step failures concentrate on odd C count / the wrong mirror class?
5. How do I placement and slot count change these error patterns?
6. Are high-cell / exact-wrong outputs stable wrong copying maps or input-dependent?

These are behavioural descriptions of outputs, NOT mechanistic claims.

**When it finishes, send back `AR_Identity_Diagnostics_Review.ipynb`.** The last cell prints its path and offers a
download. It is a self-contained snapshot with every table and figure baked in.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION ------------------------------------------------------------------------------------------------
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
ARTIFACT_ROOTS = [f"{RESULTS_ROOT}/identity_varlen_02345_all_models",     # S=5/train02345 (+ variable_length)
                  f"{RESULTS_ROOT}/identity_ar_count_extrapolation",      # S=5/train02, train023, train0234
                  f"{RESULTS_ROOT}/identity_ar_slot_count"]               # S=2/3/4/6 (notebook 16)
RUN_OVERRIDES = {}   # e.g. {"S5/train02/transformer_ar": "/content/drive/MyDrive/.../eval/transformer_ar/ARTIFACT.json"}
INCLUDE_VARIABLE_LENGTH = True        # optional history; never blocks the core analysis
RECOMPUTE_MISSING_PREDICTIONS = False # no training / re-evaluation is performed by this notebook
MAX_NEW_INFERENCE_RECORDS_PER_RUN = 30000
VERIFY_PREDICTION_SHA = True
NEAREST_PER_WORD = 20                 # six-step nearest-transform subset (stable hash), per word per model
OUT_ROOT = f"{RESULTS_ROOT}/ar_identity_mirror_diagnostics"
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if RECOMPUTE_MISSING_PREDICTIONS:
    raise NotImplementedError("Bounded re-inference is not implemented in this version; missing sources are reported "
                              "as unavailable instead (no checkpoint is run).")
import pandas as pd
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
from decompositionality.mirror_report import Diagnostics
D = Diagnostics(RESULTS_ROOT, ARTIFACT_ROOTS, RUN_OVERRIDES, OUT_ROOT, INCLUDE_VARIABLE_LENGTH, VERIFY_PREDICTION_SHA,
                NEAREST_PER_WORD)
D.conventions()


In [ ]:
# --- 1. Discover runs (read-only) and audit sources -------------------------------------------------------
D.discover()

In [ ]:
# --- 2. Load predictions, audits, exact group matching (cached by prediction sha), overview ------------------
D.load()
D.overview()

In [ ]:
# --- 3. Singleton diagnostics -----------------------------------------------------------------------------
D.singletons()

In [ ]:
# --- 4. Same-target comparisons (A~ACC, B~BCC, C~CCC, C~CCCCC) and identity controls ----------------------
D.equivalence()

In [ ]:
# --- 5. Held-out source-position copying maps ---------------------------------------------------------------
D.source_maps()

In [ ]:
# --- 6. S=6 six-operation diagnosis ------------------------------------------------------------------------
D.six_step()

In [ ]:
# --- 7. Placement / slot-count / optional format comparisons -------------------------------------------------
D.comparisons()

In [ ]:
# --- 8. Evidence table for the six questions ----------------------------------------------------------------
D.evidence()

In [ ]:
# --- 9. Exports + self-contained review notebook (SEND THIS FILE BACK) -----------------------------------------
rp = D.export()
print('REVIEW FILE TO SEND BACK:', rp)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp))